# Hankel-Rank-Regularised DQN (HR-DQN) on ALE/MsPacman-v5

This notebook loads [config_hankel.yaml](config_hankel.yaml) and benchmarks
`HankelDQNAgent` — the classical (Double-)DQN loss plus a **truncated-nuclear-norm
penalty on Hankel matrices of predicted Q-values along replayed sub-trajectory
windows** (see [docs/hankel_regularised_dqn.md](../../docs/hankel_regularised_dqn.md)
for the maths). At `hankel_weight: 0` the agent reproduces `QAgent` training exactly,
so the `baseline` variant *is* the classical DQN through the identical pipeline.

**Why Atari.** On CartPole/Acrobot the on-policy Q-trace Hankel collapses to low
rank early on its own, leaving the penalty little to bite on (see the preliminary
Acrobot results). Ms. Pac-Man's value landscape is far richer, so the natural
low-rankness should settle much later — the hypothesis under test is that the
penalty plays a more significant role here, acting as real pressure toward
low-order value dynamics instead of confirming an already-low-rank solution.
Watch `diag_batch_eff_rank` for the `baseline` run first: if it stays high for
long stretches, the premise holds.

Grid: `baseline` (λ permanently 0 via an unreachable warm-up clock — training is
classical DQN, but the window rank diagnostics are still computed under
`no_grad` each grad step, so the premise curve above actually gets recorded; a
literal `hankel_weight=0` run would save all-NaN diagnostics), `config`
(config_hankel.yaml as-is: always-on **ungated** penalty, r=2, λ=1e-2,
`gate_threshold: null` — a toy run showed the campaign's ρ=0.25 gate excludes
every window on Atari), and `progress` (the campaign winner's full recipe from
[docs/hankel_speedup_campaign.md](../../docs/hankel_speedup_campaign.md)
transferred to Ms. Pac-Man: r=2, λ=1e-2, gate ρ=0.25, latched on once the
rolling-10 life return crosses 200, ramp 2000 from the latch —
note it keeps the gate that `config` drops) × seeds, cached as
`results_hankel/<variant>_s<seed>.npz`. The cache is config-aware: each npz
stores the resolved config that produced it, and a run re-runs automatically
when that no longer matches the current yaml + overrides (deleting a file still
forces a re-run). Note a base-config edit invalidates *all* variants —
`baseline`/`progress` only pin their hankel keys and inherit the rest. Runs log
live to `runs/<variant>_s<seed>/` for the result viewer app.

**Cost warning:** at the config's full `no_episodes` a single run is days of
GPU time; the grid multiplies that by variants × seeds. `SEEDS` below and
`training.no_episodes` in the yaml are the knobs — trim them for a pilot before
committing to the full grid.

## Imports

In [ ]:
import csv, json, pathlib, random, shutil, sys, time

import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml

import ale_py
import gymnasium as gym

gym.register_envs(ale_py)  # make the ALE/... env ids visible to gym.make

SRC = pathlib.Path.cwd().parents[1] / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from experiment import load_config, build_env, build_agent, train, make_run_logger
from agents.hankel_dqn_agent import HankelDQNAgent
from agents.hankel_regulariser import HankelRankPenalty
from analysis.low_rank.hankel_policy import collect_hankel_sequences, _hankel_from_sequence
from analysis.low_rank.rank import compute_rank_metrics
from analysis.run_logger import RunLogger
from training import _greedy_episode_return

## Config

In [ ]:
cfg = load_config("config_hankel.yaml")
print("device:", cfg["experiment"]["_device"])
cfg["agent"]

## Environment and Q-network

In [ ]:
env0 = build_env(cfg)
print("obs shape:", env0.observation_space.shape, "n_actions:", env0.action_space.n)
env0.close()

In [ ]:
class NatureCNN(torch.nn.Module):
    """Maps a (C, 84, 84) frame stack to Q-values of shape (n_actions,).
    Built by the agent via q_network(**nn_extra_kwargs); uint8 frames are
    normalised to [0, 1] inside forward (scale_obs=False keeps the buffer uint8)."""
    def __init__(self, in_channels, n_actions, fc_hidden=512):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels, 32, kernel_size=8, stride=4), torch.nn.ReLU(),
            torch.nn.Conv2d(32, 64, kernel_size=4, stride=2),          torch.nn.ReLU(),
            torch.nn.Conv2d(64, 64, kernel_size=3, stride=1),          torch.nn.ReLU(),
            torch.nn.Flatten(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, 84, 84)
            flat_dim = self.features(dummy).shape[1]
        self.head = torch.nn.Sequential(
            torch.nn.Linear(flat_dim, fc_hidden), torch.nn.ReLU(),
            torch.nn.Linear(fc_hidden, n_actions),
        )

    def forward(self, x):
        x = x.float() / 255.0
        return self.head(self.features(x))

## Benchmark: variants × seeds (cached)

In [ ]:
ENV_NAME = "ALE/MsPacman-v5"

In [ ]:
VARIANTS = {
    # Diagnostics-only classical DQN: the warm-up clock never elapses, so
    # lambda_eff stays 0 and the penalty never touches the loss (parameter-
    # identical per train call to hankel_weight=0 — tests/test_hankel_dqn.py::
    # test_lambda0_matches_disabled_penalty), but the window diagnostics
    # (diag_batch_eff_rank etc.) are recorded under no_grad — required for the
    # premise check, which a literal hankel_weight=0 run cannot provide (its
    # diag arrays are all-NaN). Costs the same per-step SVD as the other arms,
    # and its RNG draws differ from a literal weight-0 run (same distribution).
    "baseline": dict(hankel_weight=1e-2, warmup_grad_steps=10**9),
    "config":   dict(),                    # config_hankel.yaml as-is (always-on, ungated)
    # The Acrobot speed-campaign winner's full recipe (docs/hankel_speedup_campaign.md,
    # round 5) transferred as-is — unlike `config` it KEEPS the spectral gate
    # (rho=0.25): the penalty latches on only once the rolling-10 return crosses
    # the threshold, with a 2000-grad-step ramp from the latch (the latch
    # replaces the warm-up clock).
    # Threshold is a life-return: per-FIRST-LIFE score (terminal_on_life_loss=True makes an
    # episodic-buffer episode one life, and each reset restarts the game, so the
    # tracked return is the first life of a fresh game). Measured random play:
    # ~120 per first life (rolling-10 means 100-150), so 200 marks the first
    # real learning. First guess — tune from baseline curves.
    "progress": dict(hankel_weight=1e-2, hankel_order=2, gate_threshold=0.25,
                     warmup_grad_steps=0, ramp_grad_steps=2000,
                     engage_reward_threshold=200, engage_reward_window=10),
}
SEEDS = [0, 1]  # Atari runs are expensive; extend once the pilot grid looks sane
RESULTS = pathlib.Path("results_hankel")
RESULTS.mkdir(exist_ok=True)


def resolve_cfg(overrides, seed):
    """The exact config a benchmark run uses (yaml + variant overrides + seed)."""
    cfg = load_config("config_hankel.yaml")
    cfg["experiment"]["seed"] = seed
    cfg["agent"].update(overrides)
    # No rank/spectra analysis inside benchmark runs, but keep a 25-episode tick
    # so rewards.csv / checkpoints refresh and the run is watchable live in the
    # result viewer app (runs/<variant>_s<seed>/).
    cfg["analysis"] = {"ep_freq": 25, "methods": []}
    return cfg


def cache_key(cfg):
    """Canonical JSON of every config section that determines the run's outcome
    (device is deliberately excluded), stored inside the npz so a run re-runs
    automatically when the config that produced it no longer matches."""
    parts = {k: cfg[k] for k in ("environment", "network", "agent", "training")}
    parts["seed"] = cfg["experiment"]["seed"]
    return json.dumps(parts, sort_keys=True, default=str)


def is_cached(out_path, key):
    if not out_path.exists():
        return False
    with np.load(out_path) as d:
        return "cfg_json" in d.files and str(d["cfg_json"]) == key


def run_one(cfg, out_path, run_id, key):
    seed = cfg["experiment"]["seed"]
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    env = build_env(cfg)
    nn_extra = {"in_channels": env.observation_space.shape[0],
                "n_actions": env.action_space.n,
                "fc_hidden": cfg["network"]["fc_hidden"]}
    agent = build_agent(cfg, env, q_network=NatureCNN, nn_extra_kwargs=nn_extra,
                        agent_cls=HankelDQNAgent)
    diags = []
    def train_hook(_orig=agent.train):
        d = _orig()
        if d is not None:
            diags.append(d)
        return d
    agent.train = train_hook

    run_dir = pathlib.Path.cwd() / "runs" / run_id
    if run_dir.exists():
        shutil.rmtree(run_dir)  # stale logs from an interrupted/old-config run
    logger = RunLogger(pathlib.Path.cwd(), config_path="config_hankel.yaml", run_id=run_id)
    # RunLogger copies the yaml verbatim; overwrite with the *resolved* config so
    # runs/<id>/config.yaml records the variant overrides (the result viewer app
    # reads this file — without this, baseline/progress runs display the wrong config).
    with open(logger.dir / "config.yaml", "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    rewards = train(cfg, agent, env, run_logger=logger)

    eps = agent.epsilon
    agent.epsilon = 0.0  # greedy eval + on-policy probe
    evals = [_greedy_episode_return(agent, env, seed=30_000 + i) for i in range(20)]
    seqs = collect_hankel_sequences(agent, env, seed=777)
    eff_rank_q = compute_rank_metrics(_hankel_from_sequence(np.asarray(seqs["Hankel Q"])))[0]
    agent.epsilon = eps

    with open(logger.dir / "eval.csv", "w", newline="") as f:  # viewer eval tile
        w = csv.writer(f)
        w.writerow(["episode", "reward"])
        w.writerows(enumerate(evals))

    diag_arrays = ({f"diag_{k}": np.array([d[k] for d in diags], float) for k in diags[0]}
                   if diags else {})
    np.savez(out_path, rewards=np.array(rewards, float), evals=np.array(evals, float),
             eff_rank_q=eff_rank_q, nan_skips=agent.nan_skips, cfg_json=key,
             **diag_arrays)
    env.close()


for variant, ov in VARIANTS.items():
    for seed in SEEDS:
        out = RESULTS / f"{variant}_s{seed}.npz"
        cfg = resolve_cfg(ov, seed)
        key = cache_key(cfg)
        if is_cached(out, key):
            print("cached:", out.name)
            continue
        if out.exists():
            print("config changed, re-running:", out.name)
        t0 = time.time()
        run_one(cfg, out, run_id=f"{variant}_s{seed}", key=key)
        print(f"{out.name}: {time.time() - t0:.0f}s")

## Learning curves

In [ ]:
data = {v: [np.load(RESULTS / f"{v}_s{s}.npz") for s in SEEDS] for v in VARIANTS}
COLORS = {"baseline": "tab:grey", "config": "tab:red", "progress": "tab:green"}

In [ ]:
plt.figure(figsize=(9, 4.5))
for v, runs in data.items():
    L = min(len(r["rewards"]) for r in runs)
    R = np.stack([np.convolve(r["rewards"][:L], np.ones(10) / 10, "valid") for r in runs])
    m = R.mean(0)
    plt.plot(m, color=COLORS[v], label=v)
    plt.fill_between(range(len(m)), R.min(0), R.max(0), color=COLORS[v], alpha=0.15)
plt.xlabel("episode"); plt.ylabel("reward (rolling mean of 10)")
plt.title(f"{ENV_NAME}: HR-DQN variants, {len(SEEDS)} seeds (band = min/max)")
plt.legend(); plt.show()

## Final evaluation — 20 greedy episodes per run

In [ ]:
print(f"{'variant':10s} {'eval20 mean±std':>22s} {'episodes to stop':>18s} {'final Hankel-Q eff-rank':>25s} {'nan_skips':>10s}")
for v, runs in data.items():
    means = np.array([r["evals"].mean() for r in runs])
    eps_used = [len(r["rewards"]) for r in runs]
    ranks = [int(r["eff_rank_q"]) for r in runs]
    skips = sum(int(r["nan_skips"]) for r in runs)
    print(f"{v:10s} {means.mean():13.1f} ± {means.std():5.1f} {str(eps_used):>18s} {str(ranks):>25s} {skips:>10d}")

## Regulariser forensics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for v, runs in data.items():
    for ax, key, lab in zip(axes, ["diag_batch_eff_rank", "diag_penalty_raw", "diag_gate_frac"],
                            ["penalty-batch eff-rank", "raw penalty (rel tail)",
                             "gate_frac (above ρ) / converged_frac (dotted)"]):
        r = runs[0]
        if key in r.files and not np.all(np.isnan(r[key])):
            ax.plot(r[key], color=COLORS[v], label=v, lw=0.8)
            if key == "diag_gate_frac" and "diag_converged_frac" in r.files:
                ax.plot(r["diag_converged_frac"], color=COLORS[v], ls=":", lw=0.8)
        ax.set_title(lab); ax.set_xlabel("train() call")
axes[0].axhline(2, ls="--", c="k", lw=0.8)
axes[0].legend(fontsize=8)
plt.suptitle("Regulariser forensics (seed 0): does the penalty lower the rank it measures?")
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 3.5))
for v, runs in data.items():
    r = runs[0]
    if "diag_td_loss" in r.files:
        plt.plot(np.convolve(r["diag_td_loss"], np.ones(10) / 10, "valid"),
                 color=COLORS[v], label=v, lw=0.9)
plt.xlabel("train() call"); plt.ylabel("TD loss (rolling 10)"); plt.legend()
plt.title("Does the penalty fight the TD objective?"); plt.show()

## Findings

*(to fill in after the grid above has run — no Atari HR-DQN reference numbers
exist yet; this is the first Ms. Pac-Man grid)*

What to check, in order:

1. **Premise** — `baseline`'s `diag_batch_eff_rank`: does the penalty-batch
   effective rank stay high for long stretches (unlike Acrobot, where it
   collapsed early on its own)? If it collapses just as fast, the "more complex
   env ⇒ loss matters more" hypothesis fails at step one.
2. **Engagement** — `config` runs ungated (`gate_threshold: null`), so its
   `gate_frac` is 0 by construction; check instead that `progress` (which keeps
   the campaign's ρ=0.25 gate) is not starved (`gate_frac` ≈ 1 after its latch)
   and that its latch actually fired (`diag_lambda_eff` > 0 at some point —
   if not, `progress` silently duplicated `baseline`).
3. **Interference** — TD-loss panel: does the penalty visibly fight the TD
   objective (persistently higher TD loss for `config`/`progress` vs `baseline`)?
4. **Outcome** — learning curves + eval20: any speed or final-quality separation
   at the pilot seed count is directional only.

The `progress` engagement threshold (200) is a first guess — recalibrate it
from the `baseline` learning curves before trusting the `progress` runs.

## Single instrumented run (optional)

For the full artifact trail (spectra figures, `hankel_sweep.csv` on-policy rank tracking,
`train_diagnostics.csv`, checkpoints under `runs/<timestamp>/`), run the canonical
single-experiment path with the config as-is:

```python
cfg = load_config("config_hankel.yaml")
env = build_env(cfg)
nn_extra = {"in_channels": env.observation_space.shape[0], "n_actions": env.action_space.n,
            "fc_hidden": cfg["network"]["fc_hidden"]}
agent = build_agent(cfg, env, q_network=NatureCNN, nn_extra_kwargs=nn_extra, agent_cls=HankelDQNAgent)
logger = make_run_logger(cfg, config_path="config_hankel.yaml")
rewards = train(cfg, agent, env, run_logger=logger)
```